In [4]:
import os
import re
import time
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup


BASE_PAGE = "https://insideairbnb.com/get-the-data/"
DOWNLOAD_DOMAIN = "data.insideairbnb.com"
OUTPUT_DIR = "inside_airbnb_data"

# Optional filters
CITY_FILTER = None   # Example: "New York City"
MAX_FILES = None     # Example: 20


def safe_filename(name: str) -> str:
    """Make a string safe for folder/file names."""
    name = re.sub(r"[^\w\s.-]", "", name)
    name = re.sub(r"\s+", "_", name.strip())
    return name


def get_html(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Python downloader for academic use)"
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.text


def parse_download_links(html: str):
    """
    Parse the Inside Airbnb data page and return a list of records:
    {
        "city_heading": ...,
        "file_label": ...,
        "url": ...
    }
    """
    soup = BeautifulSoup(html, "html.parser")
    records = []

    current_city = "unknown_city"

    for tag in soup.find_all(["h3", "a"]):
        if tag.name == "h3":
            current_city = tag.get_text(" ", strip=True)

        elif tag.name == "a":
            href = tag.get("href")
            if not href:
                continue

            full_url = urljoin(BASE_PAGE, href)
            parsed = urlparse(full_url)

            # Only keep actual downloadable data files
            if DOWNLOAD_DOMAIN not in parsed.netloc:
                continue

            file_label = os.path.basename(parsed.path).strip()
            if not file_label:
                continue

            records.append({
                "city_heading": current_city,
                "file_label": file_label,
                "url": full_url
            })

    return records


def filter_records(records):
    """
    Keep only listings.csv.gz files.
    """
    filtered = []

    for r in records:
        city_ok = True

        if CITY_FILTER:
            city_ok = CITY_FILTER.lower() in r["city_heading"].lower()

        file_ok = r["file_label"].lower() == "listings.csv.gz"

        if city_ok and file_ok:
            filtered.append(r)

    # Remove duplicates by URL
    seen = set()
    deduped = []
    for r in filtered:
        if r["url"] not in seen:
            seen.add(r["url"])
            deduped.append(r)

    if MAX_FILES is not None:
        deduped = deduped[:MAX_FILES]

    return deduped


def get_remote_file_size(url: str):
    """
    Try to get the remote file size with a HEAD request.
    Returns an int if available, otherwise None.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Python downloader for academic use)"
    }

    try:
        response = requests.head(url, headers=headers, allow_redirects=True, timeout=30)
        response.raise_for_status()
        content_length = response.headers.get("Content-Length")
        if content_length is not None:
            return int(content_length)
    except Exception:
        pass

    return None


def is_already_downloaded(output_path: str, url: str) -> bool:
    """
    Check whether a file already exists locally.
    If possible, compare local size to remote size.
    """
    if not os.path.exists(output_path):
        return False

    local_size = os.path.getsize(output_path)
    remote_size = get_remote_file_size(url)

    if remote_size is None:
        print(f"File exists, remote size unavailable. Skipping: {output_path}")
        return True

    if local_size == remote_size:
        print(f"File already downloaded and complete. Skipping: {output_path}")
        return True

    print(f"File exists but size differs. Re-downloading: {output_path}")
    return False


def download_file(url: str, output_path: str):
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Python downloader for academic use)"
    }

    with requests.get(url, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Reading page: {BASE_PAGE}")
    html = get_html(BASE_PAGE)

    all_records = parse_download_links(html)
    records = filter_records(all_records)

    print(f"Found {len(all_records)} downloadable links total.")
    print(f"Processing {len(records)} listings.csv.gz files after filters.")

    for i, record in enumerate(records, start=1):
        city_folder = safe_filename(record["city_heading"])
        target_dir = os.path.join(OUTPUT_DIR, city_folder)
        os.makedirs(target_dir, exist_ok=True)

        output_path = os.path.join(target_dir, record["file_label"])

        try:
            if is_already_downloaded(output_path, record["url"]):
                continue

            print(f"[{i}/{len(records)}] Downloading: {record['url']}")
            download_file(record["url"], output_path)
            time.sleep(0.5)

        except Exception as e:
            print(f"[{i}/{len(records)}] Failed: {record['url']}")
            print(f"Reason: {e}")

    print("Done.")


if __name__ == "__main__":
    main()

Reading page: https://insideairbnb.com/get-the-data/
Found 833 downloadable links total.
Processing 119 listings.csv.gz files after filters.
File already downloaded and complete. Skipping: inside_airbnb_data/Albany_New_York_United_States/listings.csv.gz
File already downloaded and complete. Skipping: inside_airbnb_data/Amsterdam_North_Holland_The_Netherlands/listings.csv.gz
File already downloaded and complete. Skipping: inside_airbnb_data/Antwerp_Flemish_Region_Belgium/listings.csv.gz
File already downloaded and complete. Skipping: inside_airbnb_data/Asheville_North_Carolina_United_States/listings.csv.gz
File already downloaded and complete. Skipping: inside_airbnb_data/Athens_Attica_Greece/listings.csv.gz
File already downloaded and complete. Skipping: inside_airbnb_data/Austin_Texas_United_States/listings.csv.gz
File already downloaded and complete. Skipping: inside_airbnb_data/Bangkok_Central_Thailand_Thailand/listings.csv.gz
File already downloaded and complete. Skipping: inside_a